In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Let's build an example process and some algorithms
#Jeremy Kedziora
#4 October 2023
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#import useful libraries
#@@@@@@@@@@@@@@@@@@@@@@@
import numpy as np
import pandas as pd

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
#a class for the environment
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
class random_walk:
    '''A class to manage the walk'''
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config
        self.states = list(range(config['n']))
        self.state = int(config['n']/2.0)

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def end_check(self):
        if self.state == 0 or self.state == self.config['n']-1:
            return True
        else:
            return False

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset(self):
        self.state = int(self.config['n']/2.0)

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def r(self):
#         if self.state == self.config['n'] - 1:
#             return 1
#         elif self.state == 0:
#             return -1
#         else:
#             return 0
        if self.state == self.config['n'] - 1:
            return 100
        elif self.state == 0:
            return -10
        else:
            return self.state

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def step(self,a):
        self.state = self.state + a
        done = self.end_check()
        return {'state':self.state,'reward':self.r(),'done':done}


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
#a class for the environment
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
class agent:
    '''A class to manage the agent'''
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config
        self.Q = {}
        for s in list(range(self.config['n'])):
            self.Q[str(s)] = {}
            for a in [-1,1]:
                self.Q[str(s)][str(a)] = round(np.random.uniform(),2)
        self.Q['0']['-1'] = 0
        self.Q['0']['1'] = 0
        self.Q[str(self.config['n']-1)]['-1'] = 100
        self.Q[str(self.config['n']-1)]['1'] = 100
        
        if self.config['double_Q']:
            self.Q_2 = {}
            for s in list(range(self.config['n'])):
                self.Q_2[str(s)] = {}
                for a in [-1,1]:
                    self.Q_2[str(s)][str(a)] = round(np.random.uniform(),2)
            self.Q_2['0']['-1'] = 0
            self.Q_2['0']['1'] = 0
            self.Q_2[str(self.config['n']-1)]['-1'] = 100
            self.Q_2[str(self.config['n']-1)]['1'] = 100


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def pi(self,s):
        if np.random.uniform() < self.config['epsilon']:
            a = np.random.choice([-1,1])
        else:
            a = int(list(self.Q[s].keys())[np.argmax(list(self.Q[s].values()))])
        return a
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def pi_DQ(self,s):
        if np.random.uniform() < self.config['epsilon']:
            a = np.random.choice([-1,1])
        else:
            a_star = np.argmax(np.array(list(self.Q[str(s)].values())) + np.array(list(self.Q_2[str(s)].values())))
            a = int(list(self.Q[s].keys())[a_star])
        return a

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Q_SARSA(self,s,a,r_pr,s_pr,a_pr):
        self.Q[str(s)][str(a)] = (self.Q[str(s)][str(a)] 
                                    + self.config['alpha']
                                      *(r_pr + self.config['gamma']*self.Q[str(s_pr)][str(a_pr)] - self.Q[str(s)][str(a)]))    #perform the SARSA update

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Q_learning(self,s,a,r_pr,s_pr):
        
        self.Q[str(s)][str(a)] = (self.Q[str(s)][str(a)] 
                                    + self.config['alpha']
                                      *(r_pr + self.config['gamma']*max(list(walker.Q[str(s_pr)].values())) - self.Q[str(s)][str(a)]))    #perform the Q-learning update

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Expected_SARSA(self,s,a,r_pr,s_pr):

        pi = [self.config['epsilon']/2]*(len(self.Q[str(s_pr)].values()))
        pi[np.argmax(list(self.Q[str(s_pr)].values()))] = 1 - self.config['epsilon'] + self.config['epsilon']/2
        expectation = sum(np.array(pi)*np.array(list(self.Q[str(s_pr)].values())))
        self.Q[str(s)][str(a)] = (self.Q[str(s)][str(a)] 
                                    + self.config['alpha']
                                      *(r_pr + self.config['gamma']*expectation - self.Q[str(s)][str(a)]))    #perform the expected SARSA update

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Double_Q(self,s,a,r_pr,s_pr):
        if np.random.uniform() < 0.5:    #with probability 0.5...
            a_star = list(self.Q[str(s_pr)].keys())[np.argmax(list(self.Q[str(s_pr)].values()))]    #pull out the best next action from Q
            self.Q[str(s)][str(a)] = self.Q[str(s)][str(a)] + self.config['alpha']*(r_pr + self.config['gamma']*self.Q_2[str(s_pr)][a_star] - self.Q[str(s)][str(a)])   #and update Q using Q_2
        else:    #with probability 0.5...
            a_star = list(self.Q_2[str(s_pr)].keys())[np.argmax(list(self.Q_2[str(s_pr)].values()))]    #pull out the best next action from Q_2
            self.Q_2[str(s)][str(a)] = self.Q_2[str(s)][str(a)] + self.config['alpha']*(r_pr + self.config['gamma']*self.Q[str(s_pr)][a_star] - self.Q_2[str(s)][str(a)])   #and update Q_2 using Q


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Temporal difference -- SARSA
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':10}
agent_config = {'n':10,'epsilon':0.1,'gamma':0.9,'alpha':0.1,'double_Q':False}
walk = random_walk(env_config)
walker = agent(agent_config)

N = 10000

for _ in range(N):    #loop over episodes
    s = walk.state    #choose an initial state -- center of the walk
    a = walker.pi(str(s))
    go = True    #set the end condition
    while go:    #while the episode is ongoing...
        update = walk.step(a)   #choose a new state
        if update['done']:
            go = False
            walk.reset()

        r_pr = walk.r()
        s_pr = update['state']
        a_pr = walker.pi(str(s_pr))
        walker.update_Q_SARSA(s,a,r_pr,s_pr,a_pr)
        a = a_pr    #set the current action to the new action
        s = s_pr    #set the current state to the new state


In [ ]:
walker.Q

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Temporal difference -- Q-learning
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':10}
agent_config = {'n':10,'epsilon':0.1,'gamma':0.9,'alpha':0.1,'double_Q':False}
walk = random_walk(env_config)
walker = agent(agent_config)

N = 10000

for j in range(N):    #loop over episodes
    s = walk.state    #choose an initial state -- center of the walk
    go = True    #set the end condition
    while go:    #while the episode is ongoing...
        a = walker.pi(str(s))
        update = walk.step(a)   #choose a new state
        if update['done']:
            go = False
            walk.reset()

        r_pr = walk.r()
        s_pr = update['state']
        walker.update_Q_learning(s,a,r_pr,s_pr)
        s = s_pr    #set the current state to the new state


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Temporal difference -- Expected SARSA
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':10}
agent_config = {'n':10,'epsilon':0.1,'gamma':0.9,'alpha':0.1,'double_Q':False}
walk = random_walk(env_config)
walker = agent(agent_config)

N = 10000

for j in range(N):    #loop over episodes
    s = walk.state    #choose an initial state -- center of the walk
    go = True    #set the end condition
    while go:    #while the episode is ongoing...
        a = walker.pi(str(s))
        update = walk.step(a)   #choose a new state
        if update['done']:
            go = False
            walk.reset()

        r_pr = walk.r()
        s_pr = update['state']
        walker.update_Expected_SARSA(s,a,r_pr,s_pr)
        s = s_pr    #set the current state to the new state


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Temporal difference -- double Q
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':10}
agent_config = {'n':10,'epsilon':0.1,'gamma':0.9,'alpha':0.1,'double_Q':True}
walk = random_walk(env_config)
walker = agent(agent_config)

N = 10000

for j in range(N):    #loop over episodes
    s = walk.state    #choose an initial state -- center of the walk
    go = True    #set the end condition
    while go:    #while the episode is ongoing...
        a = walker.pi_DQ(str(s))
        update = walk.step(a)   #choose a new state
        if update['done']:
            go = False
            walk.reset()
        r_pr = walk.r()
        s_pr = update['state']
        walker.update_Double_Q(s,a,r_pr,s_pr)
        s = s_pr    #set the current state to the new state


In [ ]:
env_config = {'n':10}
agent_config = {'n':10,'epsilon':0.1,'gamma':0.9,'alpha':0.1,'double_Q':False}
walk = random_walk(env_config)
walker = agent(agent_config)

trajectory = {'states':[],'actions':[],'rewards':[]}
s = walk.state    #choose an initial state -- center of the walk
a = walker.pi(str(s))    #choose an initial action
trajectory['states'].append(s)

n = 5
T = np.inf    #set the stopping conditions
t = 1    #set the current turn
gammas = []
for t in range(0,n-1):
    gammas.append(agent_config['gamma']**t)
gammas
# while t < T:    #while the episode is ongoing...
#     update = walk.step(a)   #choose a new state and generate a reward
#     trajectory['actions'].append(a)
#     trajectory['states'].append(update['state'])
#     trajectory['rewards'].append(update['reward'])
#     if update['done']:
#         T = t + 1
#     else:
#         a_pr = walker.pi(str(update['state']))
#         a = a_pr
#     tau = t - n
#     t_star = min(tau + n - 1,T)
#     if tau >= 0 and tau <= T - 1:
        
#         print(trajectory['rewards'],trajectory['rewards'][tau:t_star],t,tau,t_star)
#         #print(trajectory['states'][tau],trajectory['actions'][tau],trajectory['rewards'][tau:t_star],tau,t_star)
    
#     t += 1
# #     if update['done']:
# #         go = False
# #         walk.reset()

# #     r_pr = walk.r()
# #     s_pr = update['state']
# #     walker.update_Q_learning(s,a,r_pr,s_pr)
# #     s = s_pr    #set the current state to the new state


In [ ]:
trajectory['rewards']

In [ ]:
t_star - tau -1